# Estabilidad de bases ENIGH 2018-2024

En este notebook reviso únicamente qué variables puedo comparar de forma consistente entre 2018, 2020, 2022 y 2024. No incluyo distribuciones numéricas, categóricas, ingresos, correlaciones ni pairplot.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 180)
pd.set_option("display.max_colwidth", 180)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_ROOT = ROOT / "data/raw/EINGH"
INTERIM_ROOT = ROOT / "data/interim/revision_1"
YEARS = [2018, 2020, 2022, 2024]
CORE_TABLES = ["concentradohogar", "hogares", "ingresos", "poblacion", "trabajos", "viviendas"]

HOMOLOGATION_RENAMES = {
    "hogares": {2024: {"anio_carre": "anio_carret", "num_carre": "num_carret", "anio_pick": "anio_pickup", "num_pick": "num_pickup"}},
    "viviendas": {2024: {"medid_luz": "medidor_luz", "combus": "combustible"}},
    "concentradohogar": {2024: {"aten_hosp": "hospital"}},
}

KNOWN_NOT_COMPARABLE = {
    ("concentradohogar", "atenc_ambu"): "No la homologo con `ambul_serv` porque cambia la definición.",
    ("concentradohogar", "ambul_serv"): "No la homologo con `atenc_ambu` porque cambia la definición.",
    ("concentradohogar", "medicinas"): "No la homologo con `medic_prod` porque cambia de medicamentos sin receta a medicamentos y productos sanitarios.",
    ("concentradohogar", "medic_prod"): "No la homologo con `medicinas` porque cambia de medicamentos sin receta a medicamentos y productos sanitarios.",
    ("hogares", "anio_grab"): "No la homologo con `anio_table` porque radiograbadora y tablet no representan el mismo bien.",
    ("hogares", "anio_table"): "No la homologo con `anio_grab` porque radiograbadora y tablet no representan el mismo bien.",
    ("hogares", "er_otro"): "No la homologo con `f_otro` porque radio por otro medio y afectación climática no son equivalentes.",
    ("hogares", "f_otro"): "No la homologo con `er_otro` porque radio por otro medio y afectación climática no son equivalentes.",
    ("viviendas", "focos_inca"): "No la homologo con `focos` porque focos incandescentes y total de focos no miden lo mismo.",
    ("viviendas", "focos"): "No la homologo con `focos_inca` porque focos incandescentes y total de focos no miden lo mismo.",
    ("viviendas", "estufa_chi"): "Necesito catálogo antes de homologarla con `fogon_chi` porque la definición parece ampliarse.",
    ("viviendas", "fogon_chi"): "Necesito catálogo antes de homologarla con `estufa_chi` porque la definición parece ampliarse.",
}


def clean_series(s):
    if pd.api.types.is_object_dtype(s) or str(s.dtype).startswith("string"):
        return s.astype("string").str.strip().replace({"": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    return s


def renamed_header(table, year):
    cols = list(pd.read_csv(RAW_ROOT / str(year) / f"{table}.csv", nrows=0, encoding="utf-8-sig").columns)
    mapping = HOMOLOGATION_RENAMES.get(table, {}).get(year, {})
    return [mapping.get(col, col) for col in cols]


def years_label(years):
    return ", ".join(str(year) for year in years)


def pattern_label(years, no_real_info=False):
    if no_real_info:
        return "2018, 2020, 2022, 2024; sin información real en algún año"
    if years == [2018]:
        return "Solo 2018"
    if years == [2024]:
        return "Solo 2024"
    return years_label(years)


def exclusion_reason(table, variable, years, counts=None):
    if counts is not None:
        empty_years = [str(year) for year, n in counts.items() if n == 0]
        return f"La columna existe, pero no tiene información real en {', '.join(empty_years)}."
    if (table, variable) in KNOWN_NOT_COMPARABLE:
        return KNOWN_NOT_COMPARABLE[(table, variable)]
    if years == [2024]:
        return "Solo aparece en 2024."
    if years == [2018]:
        return "Solo aparece en 2018."
    if years == [2018, 2020, 2022]:
        return "No aparece en 2024."
    if years == [2020, 2022, 2024]:
        return "No aparece en 2018."
    if years == [2022, 2024]:
        return "Solo aparece en 2022 y 2024."
    if years == [2020, 2022]:
        return "Solo aparece en 2020 y 2022."
    return f"Solo aparece en {years_label(years)}."


raw_headers = {table: {year: renamed_header(table, year) for year in YEARS} for table in CORE_TABLES}
stack_paths = {table: INTERIM_ROOT / f"{table}_common_2018_2024.csv.gz" for table in CORE_TABLES}

## 1. Variables comparables en los cuatro años

Para considerar una variable como longitudinal exijo que esté presente después de la homologación, que conserve un significado comparable y que tenga información real en cada levantamiento.

In [ ]:
confirmed_mappings = []
for table, by_year in HOMOLOGATION_RENAMES.items():
    for year, mapping in by_year.items():
        for source, target in mapping.items():
            confirmed_mappings.append({"tabla": table, "año": year, "nombre_original": source, "nombre_homologado": target})
confirmed_mappings = pd.DataFrame(confirmed_mappings)
display(confirmed_mappings)

comparable_by_table = {}
no_real_info = {}
missing_rows = []
year_cells = {year: {"missing": 0, "total": 0} for year in YEARS}

for table in CORE_TABLES:
    df = pd.read_csv(stack_paths[table], low_memory=False)
    comparable_by_table[table] = []
    no_real_info[table] = []
    for col in [c for c in df.columns if c != "anio"]:
        s = clean_series(df[col])
        counts = s.groupby(df["anio"]).apply(lambda x: int(x.notna().sum())).reindex(YEARS, fill_value=0)
        if (counts > 0).all():
            comparable_by_table[table].append(col)
            missing_rows.append({"tabla": table, "variable": col, "missing_pct": float(s.isna().mean() * 100)})
        else:
            no_real_info[table].append((col, counts.to_dict()))
    for year, g in df.groupby("anio"):
        cols = comparable_by_table[table]
        data = g[cols].copy()
        for col in cols:
            data[col] = clean_series(data[col])
        year_cells[int(year)]["missing"] += int(data.isna().sum().sum())
        year_cells[int(year)]["total"] += int(data.shape[0] * len(cols))

comparable_summary = pd.DataFrame([
    {"Tabla": table, "Número de variables comparables": len(cols), "Variables": ", ".join(cols[:20]) + (" ..." if len(cols) > 20 else "")}
    for table, cols in comparable_by_table.items()
])
display(comparable_summary)
print(f"Total de variables comparables: {sum(len(cols) for cols in comparable_by_table.values())}")

### Variables que voy a utilizar en análisis longitudinales

**concentradohogar** (124): `folioviv`, `foliohog`, `ubica_geo`, `tam_loc`, `est_socio`, `est_dis`, `upm`, `factor`, `clase_hog`, `sexo_jefe`, `edad_jefe`, `educa_jefe`, `tot_integ`, `hombres`, `mujeres`, `mayores`, `menores`, `p12_64`, `p65mas`, `ocupados`, `percep_ing`, `perc_ocupa`, `ing_cor`, `ingtrab`, `trabajo`, `sueldos`, `horas_extr`, `comisiones`, `aguinaldo`, `indemtrab`, `otra_rem`, `remu_espec`, `negocio`, `noagrop`, `industria`, `comercio`, `servicios`, `agrope`, `agricolas`, `pecuarios`, `reproducc`, `pesca`, `otros_trab`, `rentas`, `utilidad`, `arrenda`, `transfer`, `jubilacion`, `becas`, `donativos`, `remesas`, `bene_gob`, `transf_hog`, `trans_inst`, `estim_alqu`, `otros_ing`, `gasto_mon`, `alimentos`, `ali_dentro`, `cereales`, `carnes`, `pescado`, `leche`, `huevo`, `aceites`, `tuberculo`, `verduras`, `frutas`, `azucar`, `cafe`, `especias`, `otros_alim`, `bebidas`, `ali_fuera`, `tabaco`, `vesti_calz`, `vestido`, `calzado`, `vivienda`, `alquiler`, `pred_cons`, `agua`, `energia`, `limpieza`, `cuidados`, `utensilios`, `enseres`, `salud`, `hospital`, `transporte`, `publico`, `foraneo`, `adqui_vehi`, `mantenim`, `refaccion`, `combus`, `comunica`, `educa_espa`, `educacion`, `esparci`, `paq_turist`, `personales`, `cuida_pers`, `acces_pers`, `otros_gas`, `transf_gas`, `percep_tot`, `retiro_inv`, `prestamos`, `otras_perc`, `ero_nm_viv`, `ero_nm_hog`, `erogac_tot`, `cuota_viv`, `mater_serv`, `material`, `servicio`, `deposito`, `prest_terc`, `pago_tarje`, `deudas`, `balance`, `otras_erog`, `smg`.

**hogares** (123): `folioviv`, `foliohog`, `huespedes`, `huesp_come`, `num_trab_d`, `trab_come`, `acc_alim1`, `acc_alim2`, `acc_alim3`, `acc_alim4`, `acc_alim5`, `acc_alim6`, `acc_alim7`, `acc_alim8`, `acc_alim9`, `acc_alim10`, `acc_alim11`, `acc_alim12`, `acc_alim13`, `acc_alim14`, `acc_alim15`, `acc_alim16`, `alim17_1`, `alim17_2`, `alim17_3`, `alim17_4`, `alim17_5`, `alim17_6`, `alim17_7`, `alim17_8`, `alim17_9`, `alim17_10`, `alim17_11`, `alim17_12`, `acc_alim18`, `telefono`, `celular`, `tv_paga`, `conex_inte`, `num_auto`, `anio_auto`, `num_van`, `anio_van`, `num_pickup`, `anio_pickup`, `num_moto`, `anio_moto`, `num_bici`, `anio_bici`, `num_trici`, `anio_trici`, `num_carret`, `anio_carret`, `num_canoa`, `anio_canoa`, `num_otro`, `anio_otro`, `num_ester`, `anio_ester`, `num_radio`, `anio_radio`, `num_tva`, `anio_tva`, `num_tvd`, `anio_tvd`, `num_dvd`, `anio_dvd`, `num_licua`, `anio_licua`, `num_tosta`, `anio_tosta`, `num_micro`, `anio_micro`, `num_refri`, `anio_refri`, `num_estuf`, `anio_estuf`, `num_lavad`, `anio_lavad`, `num_planc`, `anio_planc`, `num_maqui`, `anio_maqui`, `num_venti`, `anio_venti`, `num_aspir`, `anio_aspir`, `num_compu`, `anio_compu`, `num_impre`, `anio_impre`, `num_juego`, `anio_juego`, `tsalud1_h`, `tsalud1_m`, `habito_1`, `habito_2`, `habito_3`, `habito_4`, `habito_5`, `habito_6`, `consumo`, `tarjeta`, `pagotarjet`, `regalotar`, `regalodado`, `autocons`, `regalos`, `remunera`, `transferen`, `parto_g`, `negcua`, `est_alim`, `est_trans`, `bene_licon`, `cond_licon`, `lts_licon`, `otros_lts`, `diconsa`, `frec_dicon`, `cond_dicon`, `pago_dicon`, `otro_pago`.

**ingresos** (17): `folioviv`, `foliohog`, `numren`, `clave`, `mes_1`, `mes_2`, `mes_3`, `mes_4`, `mes_5`, `mes_6`, `ing_1`, `ing_2`, `ing_3`, `ing_4`, `ing_5`, `ing_6`, `ing_tri`.

**poblacion** (161): `folioviv`, `foliohog`, `numren`, `parentesco`, `sexo`, `edad`, `madre_hog`, `madre_id`, `padre_hog`, `padre_id`, `hablaind`, `lenguaind`, `hablaesp`, `comprenind`, `etnia`, `alfabetism`, `asis_esc`, `nivel`, `grado`, `tipoesc`, `tiene_b`, `otorg_b`, `forma_b`, `tiene_c`, `otorg_c`, `forma_c`, `nivelaprob`, `gradoaprob`, `antec_esc`, `residencia`, `edo_conyug`, `pareja_hog`, `conyuge_id`, `segsoc`, `ss_aa`, `ss_mm`, `redsoc_1`, `redsoc_2`, `redsoc_3`, `redsoc_4`, `redsoc_5`, `redsoc_6`, `hor_1`, `min_1`, `usotiempo1`, `hor_2`, `min_2`, `usotiempo2`, `hor_3`, `min_3`, `usotiempo3`, `hor_4`, `min_4`, `usotiempo4`, `hor_5`, `min_5`, `usotiempo5`, `hor_6`, `min_6`, `usotiempo6`, `hor_7`, `min_7`, `usotiempo7`, `hor_8`, `min_8`, `usotiempo8`, `atemed`, `inst_1`, `inst_2`, `inst_3`, `inst_4`, `inst_5`, `inst_6`, `inscr_1`, `inscr_2`, `inscr_3`, `inscr_4`, `inscr_5`, `inscr_6`, `inscr_7`, `inscr_8`, `prob_anio`, `prob_mes`, `prob_sal`, `aten_sal`, `servmed_1`, `servmed_2`, `servmed_3`, `servmed_4`, `servmed_5`, `servmed_6`, `servmed_7`, `servmed_8`, `servmed_9`, `servmed_10`, `servmed_11`, `hh_lug`, `mm_lug`, `hh_esp`, `mm_esp`, `pagoaten_1`, `pagoaten_2`, `pagoaten_3`, `pagoaten_4`, `pagoaten_5`, `pagoaten_6`, `pagoaten_7`, `noatenc_1`, `noatenc_2`, `noatenc_3`, `noatenc_4`, `noatenc_5`, `noatenc_6`, `noatenc_7`, `noatenc_8`, `noatenc_9`, `noatenc_10`, `noatenc_11`, `noatenc_12`, `noatenc_13`, `noatenc_14`, `noatenc_15`, `noatenc_16`, `norecib_1`, `norecib_2`, `norecib_3`, `norecib_4`, `norecib_5`, `norecib_6`, `norecib_7`, `norecib_8`, `norecib_9`, `norecib_11`, `razon_1`, `razon_3`, `razon_4`, `razon_5`, `razon_6`, `razon_7`, `razon_8`, `razon_9`, `razon_10`, `razon_11`, `diabetes`, `pres_alta`, `peso`, `segvol_1`, `segvol_2`, `segvol_3`, `segvol_4`, `segvol_5`, `segvol_6`, `segvol_7`, `hijos_viv`, `hijos_mue`, `hijos_sob`, `trabajo_mp`, `motivo_aus`, `act_pnea1`, `act_pnea2`, `num_trabaj`.

**trabajos** (56): `folioviv`, `foliohog`, `numren`, `id_trabajo`, `trapais`, `subor`, `indep`, `personal`, `pago`, `contrato`, `tipocontr`, `pres_1`, `pres_2`, `pres_3`, `pres_4`, `pres_5`, `pres_6`, `pres_7`, `pres_8`, `pres_9`, `pres_10`, `pres_11`, `pres_12`, `pres_13`, `pres_14`, `pres_15`, `pres_16`, `pres_17`, `pres_18`, `pres_19`, `pres_20`, `htrab`, `sinco`, `scian`, `clas_emp`, `tam_emp`, `no_ing`, `tiene_suel`, `tipoact`, `socios`, `soc_nr1`, `soc_nr2`, `soc_resp`, `otra_act`, `tipoact2`, `tipoact3`, `tipoact4`, `lugar`, `conf_pers`, `medtrab_1`, `medtrab_2`, `medtrab_3`, `medtrab_4`, `medtrab_5`, `medtrab_6`, `medtrab_7`.

**viviendas** (60): `folioviv`, `tipo_viv`, `mat_pared`, `mat_techos`, `mat_pisos`, `antiguedad`, `antigua_ne`, `cocina`, `cocina_dor`, `cuart_dorm`, `num_cuarto`, `dotac_agua`, `excusado`, `uso_compar`, `sanit_agua`, `biodigest`, `bano_comp`, `bano_excus`, `bano_regad`, `drenaje`, `disp_elect`, `focos_ahor`, `combustible`, `eli_basura`, `tenencia`, `renta`, `estim_pago`, `pago_viv`, `pago_mesp`, `tipo_adqui`, `viv_usada`, `num_dueno1`, `hog_dueno1`, `num_dueno2`, `hog_dueno2`, `escrituras`, `lavadero`, `fregadero`, `regadera`, `tinaco_azo`, `cisterna`, `pileta`, `calent_sol`, `calent_gas`, `medidor_luz`, `bomba_agua`, `tanque_gas`, `aire_acond`, `calefacc`, `tot_resid`, `tot_hom`, `tot_muj`, `tot_hog`, `ubica_geo`, `tam_loc`, `est_socio`, `est_dis`, `upm`, `factor`, `procaptar`.

## 2. Distribución general de faltantes

Calculo los faltantes únicamente sobre las variables comparables en los cuatro años. Esta parte me permite revisar si después de homologar todavía queda un levantamiento con pérdida sistemática de información.

In [ ]:
missing_df = pd.DataFrame(missing_rows)
missing_labels = ["0%", ">0-5%", ">5-20%", ">20-50%", ">50-90%", ">90%"]
missing_df["Rango de faltantes"] = pd.cut(
    missing_df["missing_pct"],
    bins=[-1e-9, 0, 5, 20, 50, 90, 100],
    labels=missing_labels,
    include_lowest=True,
    right=True,
)
missing_distribution = missing_df.groupby("Rango de faltantes", observed=False).size().reindex(missing_labels, fill_value=0).reset_index(name="Número de variables")
missing_distribution["% de variables"] = missing_distribution["Número de variables"] / len(missing_df) * 100
display(missing_distribution)

top_missing = missing_df.sort_values("missing_pct", ascending=False).head(15).rename(columns={"tabla": "Tabla", "variable": "Variable", "missing_pct": "% faltantes 2018-2024"})
display(top_missing[["Tabla", "Variable", "% faltantes 2018-2024"]].round(2))

missing_by_year = pd.DataFrame([
    {"Año": year, "Celdas comparables evaluadas": values["total"], "% faltantes": values["missing"] / values["total"] * 100}
    for year, values in year_cells.items()
])
display(missing_by_year.round(2))

## 3. Variables fuera del análisis longitudinal

Aquí separo las variables que no puedo usar como longitudinales estrictas porque no aparecen en los cuatro años, porque cambiaron de definición o porque existen como columna pero no tienen información real en algún levantamiento.

In [ ]:
excluded_rows = []
for table in CORE_TABLES:
    union_cols = sorted(set().union(*[set(raw_headers[table][year]) for year in YEARS]))
    comparable = set(comparable_by_table[table])
    noinfo_map = dict(no_real_info[table])
    for variable in union_cols:
        if variable in comparable:
            continue
        years = [year for year in YEARS if variable in raw_headers[table][year]]
        counts = noinfo_map.get(variable)
        excluded_rows.append({
            "Tabla": table,
            "Variable": variable,
            "Años disponibles": years_label(years),
            "Patrón": pattern_label(years, counts is not None),
            "Motivo": exclusion_reason(table, variable, years, counts),
        })

excluded_detail = pd.DataFrame(excluded_rows).sort_values(["Tabla", "Patrón", "Variable"]).reset_index(drop=True)
exclusion_summary = pd.DataFrame([
    {
        "Tabla": table,
        "Variables consideradas": len(set().union(*[set(raw_headers[table][year]) for year in YEARS])),
        "Comparables en 4 años": len(comparable_by_table[table]),
        "Fuera del análisis longitudinal": int((excluded_detail["Tabla"] == table).sum()),
    }
    for table in CORE_TABLES
])
display(exclusion_summary)

pattern_summary = excluded_detail.groupby("Patrón").size().reset_index(name="Número de variables").sort_values("Número de variables", ascending=False)
display(pattern_summary)
display(excluded_detail)

### Variables que no coinciden entre los cuatro años

**concentradohogar**

- `atenc_ambu` (2018, 2020, 2022): No la homologo con `ambul_serv` porque cambia la definición.

- `medicinas` (2018, 2020, 2022): No la homologo con `medic_prod` porque cambia de medicamentos sin receta a medicamentos y productos sanitarios.

- `ambul_serv` (2024): No la homologo con `atenc_ambu` porque cambia la definición.

- `medic_prod` (2024): No la homologo con `medicinas` porque cambia de medicamentos sin receta a medicamentos y productos sanitarios.

**hogares**

- `anio_grab` (2018, 2020, 2022): No la homologo con `anio_table` porque radiograbadora y tablet no representan el mismo bien.

- `anio_video` (2018, 2020, 2022): No aparece en 2024.

- `embarazo_g` (2018, 2020, 2022): No aparece en 2024.

- `er_aparato` (2018, 2020, 2022): No aparece en 2024.

- `er_aplicac` (2018, 2020, 2022): No aparece en 2024.

- `er_celular` (2018, 2020, 2022): No aparece en 2024.

- `er_compu` (2018, 2020, 2022): No aparece en 2024.

- `er_otro` (2018, 2020, 2022): No la homologo con `f_otro` porque radio por otro medio y afectación climática no son equivalentes.

- `er_tv` (2018, 2020, 2022): No aparece en 2024.

- `esc_radio` (2018, 2020, 2022): No aparece en 2024.

- `num_grab` (2018, 2020, 2022): No aparece en 2024.

- `num_video` (2018, 2020, 2022): No aparece en 2024.

- `recib_tvd` (2018, 2020, 2022): No aparece en 2024.

- `nr_viv` (2018, 2020, 2022, 2024): La columna existe, pero no tiene información real en 2024.

- `entidad` (2022, 2024): Solo aparece en 2022 y 2024.

- `est_dis` (2022, 2024): Solo aparece en 2022 y 2024.

- `factor` (2022, 2024): Solo aparece en 2022 y 2024.

- `upm` (2022, 2024): Solo aparece en 2022 y 2024.

- `af_cultivo` (2024): Solo aparece en 2024.

- `af_empleo` (2024): Solo aparece en 2024.

- `af_negocio` (2024): Solo aparece en 2024.

- `af_otro` (2024): Solo aparece en 2024.

- `af_salud` (2024): Solo aparece en 2024.

- `af_trabajo` (2024): Solo aparece en 2024.

- `af_viv` (2024): Solo aparece en 2024.

- `anio_lap` (2024): Solo aparece en 2024.

- `anio_table` (2024): No la homologo con `anio_grab` porque radiograbadora y tablet no representan el mismo bien.

- `camb_clim` (2024): Solo aparece en 2024.

- `f_desliza` (2024): Solo aparece en 2024.

- `f_helada` (2024): Solo aparece en 2024.

- `f_huracan` (2024): Solo aparece en 2024.

- `f_incendio` (2024): Solo aparece en 2024.

- `f_inunda` (2024): Solo aparece en 2024.

- `f_otro` (2024): No la homologo con `er_otro` porque radio por otro medio y afectación climática no son equivalentes.

- `f_sequia` (2024): Solo aparece en 2024.

- `num_lap` (2024): Solo aparece en 2024.

- `num_table` (2024): Solo aparece en 2024.

- `peliculas` (2024): Solo aparece en 2024.

**ingresos**

- `entidad` (2022, 2024): Solo aparece en 2022 y 2024.

- `est_dis` (2022, 2024): Solo aparece en 2022 y 2024.

- `factor` (2022, 2024): Solo aparece en 2022 y 2024.

- `upm` (2022, 2024): Solo aparece en 2022 y 2024.

**poblacion**

- `norecib_10` (2018, 2020, 2022, 2024): La columna existe, pero no tiene información real en 2024.

- `razon_2` (2018, 2020, 2022, 2024): La columna existe, pero no tiene información real en 2024.

- `cau_acti` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_apren` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_brazo` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_camin` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_habla` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_oir` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_ver` (2020, 2022): Solo aparece en 2020 y 2022.

- `cau_vest` (2020, 2022): Solo aparece en 2020 y 2022.

- `norecib_12` (2020, 2022): Solo aparece en 2020 y 2022.

- `pop_insabi` (2020, 2022): Solo aparece en 2020 y 2022.

- `servmed_12` (2020, 2022): Solo aparece en 2020 y 2022.

- `c_futuro` (2020, 2022, 2024): No aparece en 2018.

- `ct_futuro` (2020, 2022, 2024): No aparece en 2018.

- `disc_acti` (2020, 2022, 2024): No aparece en 2018.

- `disc_apren` (2020, 2022, 2024): No aparece en 2018.

- `disc_brazo` (2020, 2022, 2024): No aparece en 2018.

- `disc_camin` (2020, 2022, 2024): No aparece en 2018.

- `disc_habla` (2020, 2022, 2024): No aparece en 2018.

- `disc_oir` (2020, 2022, 2024): No aparece en 2018.

- `disc_ver` (2020, 2022, 2024): No aparece en 2018.

- `disc_vest` (2020, 2022, 2024): No aparece en 2018.

- `entidad` (2022, 2024): Solo aparece en 2022 y 2024.

- `est_dis` (2022, 2024): Solo aparece en 2022 y 2024.

- `factor` (2022, 2024): Solo aparece en 2022 y 2024.

- `upm` (2022, 2024): Solo aparece en 2022 y 2024.

- `causa1` (2018): Solo aparece en 2018.

- `causa2` (2018): Solo aparece en 2018.

- `causa3` (2018): Solo aparece en 2018.

- `causa4` (2018): Solo aparece en 2018.

- `causa5` (2018): Solo aparece en 2018.

- `causa6` (2018): Solo aparece en 2018.

- `causa7` (2018): Solo aparece en 2018.

- `disc1` (2018): Solo aparece en 2018.

- `disc2` (2018): Solo aparece en 2018.

- `disc3` (2018): Solo aparece en 2018.

- `disc4` (2018): Solo aparece en 2018.

- `disc5` (2018): Solo aparece en 2018.

- `disc6` (2018): Solo aparece en 2018.

- `disc7` (2018): Solo aparece en 2018.

- `segpop` (2018): Solo aparece en 2018.

- `afrod` (2024): Solo aparece en 2024.

- `edu_ini` (2024): Solo aparece en 2024.

- `inst_7` (2024): Solo aparece en 2024.

- `inst_8` (2024): Solo aparece en 2024.

- `inst_9` (2024): Solo aparece en 2024.

- `no_asis` (2024): Solo aparece en 2024.

- `no_asisb` (2024): Solo aparece en 2024.

- `pais_nac` (2024): Solo aparece en 2024.

**trabajos**

- `entidad` (2022, 2024): Solo aparece en 2022 y 2024.

- `est_dis` (2022, 2024): Solo aparece en 2022 y 2024.

- `factor` (2022, 2024): Solo aparece en 2022 y 2024.

- `upm` (2022, 2024): Solo aparece en 2022 y 2024.

**viviendas**

- `disp_agua` (2018, 2020, 2022): No aparece en 2024.

- `estufa_chi` (2018, 2020, 2022): Necesito catálogo antes de homologarla con `fogon_chi` porque la definición parece ampliarse.

- `focos_inca` (2018, 2020, 2022): No la homologo con `focos` porque focos incandescentes y total de focos no miden lo mismo.

- `tipo_finan` (2018, 2020, 2022): No aparece en 2024.

- `ab_agua` (2024): Solo aparece en 2024.

- `agua_ent` (2024): Solo aparece en 2024.

- `agua_noe` (2024): Solo aparece en 2024.

- `calen_lena` (2024): Solo aparece en 2024.

- `finan_1` (2024): Solo aparece en 2024.

- `finan_2` (2024): Solo aparece en 2024.

- `finan_3` (2024): Solo aparece en 2024.

- `finan_4` (2024): Solo aparece en 2024.

- `finan_5` (2024): Solo aparece en 2024.

- `finan_6` (2024): Solo aparece en 2024.

- `finan_7` (2024): Solo aparece en 2024.

- `finan_8` (2024): Solo aparece en 2024.

- `focos` (2024): No la homologo con `focos_inca` porque focos incandescentes y total de focos no miden lo mismo.

- `fogon_chi` (2024): Necesito catálogo antes de homologarla con `estufa_chi` porque la definición parece ampliarse.

- `lugar_coc` (2024): Solo aparece en 2024.

- `p_electric` (2024): Solo aparece en 2024.

- `p_fractura` (2024): Solo aparece en 2024.

- `p_grietas` (2024): Solo aparece en 2024.

- `p_humedad` (2024): Solo aparece en 2024.

- `p_levanta` (2024): Solo aparece en 2024.

- `p_pandeos` (2024): Solo aparece en 2024.

- `p_tuberias` (2024): Solo aparece en 2024.

## 4. Resumen de estabilidad de las bases

Al comparar los cuatro años después de homologar nombres, me quedo con **541 variables longitudinales estrictas**. Por tabla, el conjunto queda así: `concentradohogar` 124, `hogares` 123, `ingresos` 17, `poblacion` 161, `trabajos` 56 y `viviendas` 60.

En términos de completitud, **61.37%** de las variables comparables tiene entre 0% y 5% de faltantes acumulados. Ninguna variable longitudinal estricta supera 50% de faltantes en el total apilado. El punto que todavía debo cuidar es 2024, porque concentra **40.60%** de faltantes sobre las celdas comparables evaluadas, principalmente en variables de universo condicionado.

Dejo fuera **126 variables** de un total de **667 variables consideradas**. Las exclusiones ocurren porque algunas variables aparecen solo en ciertos levantamientos, otras entran a partir de 2022 o 2024, algunas desaparecen en 2024 y otras cambian de definición. Además, excluyo `nr_viv`, `norecib_10` y `razon_2` porque aunque existen como columnas en los cuatro años, no tienen información real en 2024.

Después de esta revisión no veo un problema pendiente de nomenclatura para las variables que sí conservo: las renombradas con mapping validado ya quedan integradas bajo una sola columna final. Las variables dudosas o con cambio conceptual permanecen fuera del análisis longitudinal hasta que pueda justificar una equivalencia con catálogo.